# Examen ICCD753 Recuperación de Información

## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado

Cada estudiante recibirá un conjunto de datos. Todos contienen documentos, consultas y qrels. Se utilizarán los identificadores originales de los documentos para que los resultados puedan evaluarse.

| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |
| SciFact | [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact) | [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels) |

## 3. Requisitos del sistema

### 3.1. Preparación e indexación

El sistema deberá:

1. Cargar e indexar los documentos del corpus asignado.
2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.
3. Construir un vocabulario y un índice invertido que permita recuperar documentos.
4. Conservar la relación entre cada documento indexado y su identificador original.

La unidad de evaluación será el documento original, porque los qrels relacionan consultas con identificadores de documentos. No se debe dividir el texto en fragmentos.

### 3.2. Modelos de recuperación

Para cualquier consulta de texto, el sistema deberá producir dos rankings independientes:

- TF-IDF con similitud coseno.
- BM25.

Cada ranking deberá ordenar los documentos por puntuación de relevancia y permitir solicitar los primeros `k` resultados.

### 3.3. Servicio web

El sistema deberá estar publicado en internet y ofrecer, como mínimo, los siguientes endpoints:

**Comprobación de disponibilidad**

```http
GET /health
```

**Búsqueda**

```http
POST /search
Content-Type: application/json
```

Ejemplo de solicitud:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` deberá aceptar `bm25` y `tfidf`. Una respuesta tendrá esta estructura:

```json
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "identificador_original",
      "score": 8.42,
      "title": "Título, si está disponible",
      "snippet": "Fragmento del documento recuperado"
    }
  ]
}
```

Los resultados deberán estar ordenados de mayor a menor puntuación, no contener documentos duplicados y respetar el valor solicitado de `top_k`. El servicio debe aceptar consultas que el estudiante no conozca de antemano.

## 4. Evaluación de la recuperación

Cada estudiante debe evaluar ambos modelos con consultas de la partición `test` y sus qrels. Presentará, como mínimo:

- Precision@10.
- Recall@10.
- NDCG@10.
- Una comparación de los resultados de TF-IDF y BM25.

En el Jupyter Notebook se deberá indicar qué consultas se utilizaron y permitir reproducir los cálculos. La evaluación del docente podrá utilizar otras consultas del mismo conjunto.

## 5. Prueba del docente

El docente accederá a la URL entregada y realizará solicitudes desde un cliente de web service para comprobar:

1. La disponibilidad del servicio.
2. El funcionamiento de TF-IDF y BM25.
3. El formato, orden y validez de los resultados.
4. La correspondencia de los `document_id` con el corpus y los qrels.
5. La respuesta ante consultas no incluidas en los ejemplos de entrega.

## 6. Entregables

Cada estudiante entregará:

1. La URL pública del servicio web, accesible durante el período de revisión.
2. El código en Jupyter Notebook con los resultados de la ejecución de cada bloque de código, incluyendo los resultados de las métricas de evaluación de los modelos de recuperación.

Se evaluará que ambos modelos funcionen correctamente y que cada estudiante interprete sus resultados.

## Solución · NFCorpus

Los documentos se cargan directamente con `load_dataset("BeIR/nfcorpus", "corpus")`. El notebook obtiene las consultas y los [qrels oficiales de test](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) mediante la misma biblioteca. Para la primera ejecución, se necesita acceso a Hugging Face y `datasets` instalado en el kernel de Python 3.10 o posterior (`%pip install "datasets>=3.6,<6"`).

### Preprocesamiento e índice invertido

- Se concatena el título con el texto. Cada documento conserva su `_id` y se indexa como una sola unidad.
- Se aplica normalización Unicode NFKC y se convierten las letras a minúsculas con `casefold`. La expresión regular extrae secuencias de letras y números. Se eliminan posesivos, palabras funcionales del inglés y letras aisladas. Se conservan números para expresiones como `vitamin D3`.
- El stemming se limita a plurales regulares, como `berries → berry` y `diets → diet`. Se conservan terminaciones frecuentes en términos biomédicos, como `-is` en `arthritis`. Las reglas son heurísticas y pueden producir raíces imperfectas.
- Cada término del vocabulario apunta a un diccionario `{document_id: frecuencia}`. Las frecuencias de documento, las normas TF-IDF y las longitudes se calculan una vez y sirven a los dos modelos.

### Modelos

**TF-IDF + coseno.** Se usa frecuencia cruda e IDF suavizada $\log((N+1)/(df+1))+1$. Se multiplica el TF por el IDF y se calcula el coseno con las normas L2 del documento y de la consulta. Los términos desconocidos quedan fuera del vector de consulta.

**BM25.** Se usa $k_1=1.2$, $b=0.75$ y $\operatorname{IDF}(t)=\log(1+(N-df(t)+0.5)/(df(t)+0.5))$. La frecuencia se satura mediante $f(t,d)(k_1+1)/(f(t,d)+k_1(1-b+b|d|/\operatorname{avgdl}))$. Cada término distinto de la consulta aporta una vez. Los parámetros se fijan antes de evaluar `test`.

Los dos rankings se generan desde los postings de la consulta. Se devuelven hasta `top_k` coincidencias, ordenadas por puntuación descendente y, en empates, por ID ascendente. Una consulta sin coincidencias devuelve una lista vacía.

Ambos modelos son léxicos. TF-IDF usa un vector disperso de pesos por palabra y BM25 calcula puntuaciones a partir de frecuencias. El sistema prescinde de embeddings.

### Cargar el corpus

`cargar_nfcorpus()` obtiene los documentos desde Hugging Face. Devuelve un diccionario cuya clave es el `_id` original. Esto permite localizar un documento tanto en los rankings como en los juicios de relevancia.

In [1]:
import json
import math
import re
import unicodedata
from collections import Counter, defaultdict
from statistics import mean


MODELS = ('tfidf', 'bm25')


def cargar_nfcorpus():
    """Carga NFCorpus y devuelve los documentos por identificador original."""

    from datasets import load_dataset

    ds = load_dataset("BeIR/nfcorpus", "corpus")
    documents = {row['_id']: row for row in ds['corpus']}
    if len(documents) != 3633:
        raise ValueError('El número de documentos de NFCorpus no coincide con BEIR.')
    return documents

In [2]:
documentos = cargar_nfcorpus()
identificador, documento = next(iter(documentos.items()))
print(f'Documentos cargados: {len(documentos):,}')
print(f'Primer identificador: {identificador}')
print(f'Título: {documento["title"]}')

Documentos cargados: 3,633
Primer identificador: MED-10
Título: Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland


### Tokenizar documentos y consultas

Aplicamos `tokenizar()` al título y al texto de cada documento, y también a las consultas. La función normaliza el texto, extrae los términos, descarta palabras vacías y reduce algunos plurales. El ejemplo permite comprobar el efecto de cada decisión en los términos que llegan al índice.

In [3]:
STOPWORDS = frozenset('''a about above after again against all am an and any are as at be
because been before being below between both but by can could did do does doing down during
each few for from further had has have having he her here hers herself him himself his how i
if in into is it its itself just me more most my myself of off on once only or other our ours
ourselves out over own same she should so some such than that the their theirs them themselves
then there these they this those through to too under until up very was we were what when where
which while who whom why will with would you your yours yourself yourselves'''.split())
TOKEN = re.compile(r'[^\W_]+', re.UNICODE)


def stem(token):
    """Stemming flexivo conservador para plurales regulares del inglés."""
    if len(token) > 4 and token.endswith('ies'):
        return token[:-3] + 'y'
    if len(token) > 4 and token.endswith(('ches', 'shes', 'xes', 'sses', 'zzes')):
        return token[:-2]
    if len(token) > 3 and token.endswith('s') and not token.endswith(('ss', 'us', 'is', 'ics', 'ous')):
        return token[:-1]
    return token


def tokenizar(texto):
    """Devuelve los términos normalizados que se usarán en el índice."""
    # 1. Normalizar Unicode, mayúsculas y posesivos.
    normalizado = unicodedata.normalize('NFKC', texto).casefold()
    normalizado = re.sub(r"\b([^\W_]+)['’]s\b", r'\1', normalizado)

    # 2. Extraer términos y filtrar palabras vacías y letras aisladas.
    tokens = [token for token in TOKEN.findall(normalizado)
              if token not in STOPWORDS and (len(token) > 1 or token.isdigit())]

    # 3. Aplicar stemming a los plurales regulares previstos por las reglas.
    return [stem(token) for token in tokens]

In [4]:
texto_prueba = "The BERRIES, diets and arthritis in vitamin D3."
tokens_prueba = tokenizar(texto_prueba)
assert tokens_prueba == ['berry', 'diet', 'arthritis', 'vitamin', 'd3']
print('Texto original:', texto_prueba)
print('Tokens:', tokens_prueba)

Texto original: The BERRIES, diets and arthritis in vitamin D3.
Tokens: ['berry', 'diet', 'arthritis', 'vitamin', 'd3']


### Evaluación de recuperación

Se evalúan ambos modelos sobre las mismas consultas, seleccionadas por aparecer en el split `test` de `BeIR/nfcorpus-qrels`. Los juicios con grado mayor que cero cuentan como relevantes. Precision@10 divide los aciertos por 10 incluso si se devuelven menos resultados. Recall@10 divide los aciertos por el total de documentos relevantes en los qrels de la consulta.

NDCG@10 usa el grado original como ganancia lineal y lo descuenta por $\log_2(i+1)$ en la posición $i$. El denominador es el DCG ideal formado con los diez grados más altos. Los documentos sin juicio reciben grado cero. Cada métrica final es el promedio simple de las métricas por consulta.

### Servicio web

`GET /health` confirma que el índice está cargado. `POST /search` acepta `query`, `model` (`tfidf` o `bm25`) y `top_k`, con valor predeterminado 10. Cada resultado incluye rango, identificador original, puntuación, título y fragmento.

El paquete de despliegue contiene el mismo motor y la misma aplicación WSGI de este notebook, junto con un Dockerfile. La URL que se entregue al docente debe responder directamente por HTTPS y permitir enviar ambos tipos de consultas desde Postman.

### Carga y construcción del índice

`construir_indice()` genera los postings de cada término. También calcula la longitud de los documentos y las estadísticas que reutilizan los dos modelos. Por ejemplo, los postings de `diet` indican qué documentos contienen ese término y cuántas veces aparece en cada uno.

In [5]:
def construir_indice(documentos, k1=1.2, b=0.75):
    """Construye el índice invertido y las estadísticas de TF-IDF y BM25."""
    postings = defaultdict(dict)
    longitudes = {}

    # 1. Contar términos por documento y añadirlos a sus postings.
    for documento_id, documento in documentos.items():
        texto = ' '.join((documento.get('title') or '', documento.get('text') or ''))
        frecuencias = Counter(tokenizar(texto))
        longitudes[documento_id] = frecuencias.total()
        for termino, frecuencia in frecuencias.items():
            postings[termino][documento_id] = frecuencia

    # 2. Precalcular el ajuste de longitud de BM25.
    longitud_media = mean(longitudes.values()) if documentos else 0
    ajuste_bm25 = {
        documento_id: k1 * (1 - b + b * longitud / (longitud_media or 1))
        for documento_id, longitud in longitudes.items()
    }

    # 3. Calcular las IDF y las normas de los vectores TF-IDF.
    total = len(documentos)
    idf_tfidf, idf_bm25 = {}, {}
    normas_cuadradas = defaultdict(float)
    for termino, documentos_del_termino in postings.items():
        df = len(documentos_del_termino)
        idf_tfidf[termino] = 1 + math.log((total + 1) / (df + 1))
        idf_bm25[termino] = math.log1p((total - df + 0.5) / (df + 0.5))
        for documento_id, frecuencia in documentos_del_termino.items():
            normas_cuadradas[documento_id] += (frecuencia * idf_tfidf[termino]) ** 2

    return {
        'documentos': documentos,
        'postings': dict(postings),
        'vocabulario': frozenset(postings),
        'longitudes': longitudes,
        'longitud_media': longitud_media,
        'k1': k1,
        'ajuste_bm25': ajuste_bm25,
        'idf_tfidf': idf_tfidf,
        'idf_bm25': idf_bm25,
        'normas_tfidf': {doc_id: math.sqrt(valor) for doc_id, valor in normas_cuadradas.items()},
    }

In [6]:
indice = construir_indice(documentos)
assert indice['documentos'].keys() == documentos.keys()
print(f'Términos únicos: {len(indice["vocabulario"]):,}')
print(f'Longitud media de los documentos: {indice["longitud_media"]:.2f} tokens')
print('Primeras cinco entradas de diet (ID, frecuencia):')
print(list(indice['postings']['diet'].items())[:5])

Términos únicos: 23,553
Longitud media de los documentos: 164.06 tokens
Primeras cinco entradas de diet (ID, frecuencia):
[('MED-332', 1), ('MED-557', 1), ('MED-706', 1), ('MED-711', 6), ('MED-716', 1)]


### Buscar con BM25

`buscar_bm25()` recorre los postings de cada término de la consulta. El puntaje de un documento acumula la contribución de los términos que contiene. Un término aporta más cuando es menos común en el corpus. El efecto de repetirlo dentro de un documento disminuye con la frecuencia, y el ajuste por longitud modera la ventaja de los documentos largos.

In [7]:
def ordenar_puntajes(puntajes, top_k):
    """Devuelve hasta top_k pares (ID, puntaje), resolviendo empates por ID."""
    return sorted(puntajes.items(), key=lambda par: (-par[1], par[0]))[:top_k]


def buscar_bm25(consulta, indice, top_k=10):
    """Recupera los documentos con mayor puntaje BM25 para la consulta."""
    # 1. Conservar cada término una vez, en el orden en que aparece.
    terminos = dict.fromkeys(tokenizar(consulta))
    puntajes = defaultdict(float)
    k1 = indice['k1']

    # 2. Acumular las contribuciones BM25 solamente en los documentos que contienen el término.
    for termino in terminos:
        posting = indice['postings'].get(termino)
        if not posting:
            continue
        idf = indice['idf_bm25'][termino]
        for documento_id, frecuencia in posting.items():
            ajuste = indice['ajuste_bm25'][documento_id]
            puntajes[documento_id] += idf * frecuencia * (k1 + 1) / (frecuencia + ajuste)

    # 3. Ordenar los resultados y aplicar el corte.
    return ordenar_puntajes(puntajes, top_k)

In [8]:
consulta_prueba = 'effects of green tea on blood pressure'
ranking_bm25 = buscar_bm25(consulta_prueba, indice, top_k=3)
print(f'Consulta: {consulta_prueba}')
for posicion, (doc_id, puntaje) in enumerate(ranking_bm25, start=1):
    print(f'{posicion}. {doc_id} | {puntaje:.6f} | {documentos[doc_id]["title"]}')

Consulta: effects of green tea on blood pressure
1. MED-5047 | 17.379048 | The Association between Concentrations of Green Tea and Blood Glucose Levels
2. MED-1842 | 15.715940 | The effect of sour tea (Hibiscus sabdariffa) on essential hypertension.
3. MED-4365 | 15.114947 | Adverse effects of concentrated green tea extracts.


### Buscar con TF-IDF y similitud coseno

`buscar_tfidf()` calcula el peso de cada término conocido en la consulta. Suma el producto de pesos sobre los términos compartidos y divide entre las normas de los vectores. Así obtiene el coseno y ordena los documentos por similitud. Usamos la misma consulta de BM25 para comparar posiciones.

In [9]:
def buscar_tfidf(consulta, indice, top_k=10):
    """Recupera los documentos con mayor similitud coseno TF-IDF."""
    # 1. Construir el vector de la consulta usando el vocabulario del índice.
    vocabulario = indice['vocabulario']
    idf = indice['idf_tfidf']
    frecuencias = Counter(termino for termino in tokenizar(consulta) if termino in vocabulario)
    pesos = {termino: frecuencia * idf[termino] for termino, frecuencia in frecuencias.items()}
    norma_consulta = math.sqrt(sum(peso ** 2 for peso in pesos.values()))

    # 2. Calcular el producto escalar con los documentos que comparten términos.
    productos = defaultdict(float)
    for termino, peso in pesos.items():
        for documento_id, frecuencia in indice['postings'][termino].items():
            productos[documento_id] += peso * frecuencia * idf[termino]

    # 3. Normalizar el producto para obtener el coseno y ordenar los resultados.
    puntajes = {documento_id: producto / (norma_consulta * indice['normas_tfidf'][documento_id])
                for documento_id, producto in productos.items()}
    return ordenar_puntajes(puntajes, top_k)

In [10]:
ranking_tfidf = buscar_tfidf(consulta_prueba, indice, top_k=3)
print(f'Consulta: {consulta_prueba}')
for posicion, (doc_id, puntaje) in enumerate(ranking_tfidf, start=1):
    print(f'{posicion}. {doc_id} | {puntaje:.6f} | {documentos[doc_id]["title"]}')

Consulta: effects of green tea on blood pressure
1. MED-5047 | 0.587174 | The Association between Concentrations of Green Tea and Blood Glucose Levels
2. MED-4330 | 0.517077 | Green and black tea in relation to gynecologic cancers
3. MED-4365 | 0.510802 | Adverse effects of concentrated green tea extracts.


### Construir y probar la respuesta de la API

`formatear_resultados()` toma el ranking y añade el título, un fragmento y la posición de cada documento. `create_app()` usa esa función para devolver la misma estructura JSON que se prueba en Postman.

In [11]:
BUSCADORES = {'bm25': buscar_bm25, 'tfidf': buscar_tfidf}


def formatear_resultados(consulta, modelo, indice, top_k=10):
    """Devuelve el ranking y sus metadatos con el formato JSON de la API."""
    resultados = []
    ranking = BUSCADORES[modelo](consulta, indice, top_k)
    for posicion, (documento_id, puntaje) in enumerate(ranking, start=1):
        documento = indice['documentos'][documento_id]
        titulo = documento.get('title') or ''
        texto = ' '.join((documento.get('text') or titulo).split())
        fragmento = texto[:240].rsplit(' ', 1)[0] + '…' if len(texto) > 240 else texto
        resultados.append(dict(rank=posicion, document_id=documento_id, score=puntaje,
                               title=titulo, snippet=fragmento))
    return dict(model=modelo, results=resultados)

In [12]:
from http import HTTPStatus


def create_app(indice):
    """Crea la aplicación WSGI con /health y /search."""
    def application(environ, start_response):
        def reply(status, body, headers=()):
            data = json.dumps(body, ensure_ascii=False, allow_nan=False).encode('utf-8')
            start_response(f'{status.value} {status.phrase}', [
                ('Content-Type', 'application/json; charset=utf-8'),
                ('Content-Length', str(len(data))), *headers,
            ])
            return [data]

        path = environ.get('PATH_INFO', '')
        method = environ.get('REQUEST_METHOD', '')
        expected_method = {'/': 'GET', '/health': 'GET', '/search': 'POST'}.get(path)
        if expected_method is None:
            return reply(HTTPStatus.NOT_FOUND, {'error': 'Ruta desconocida.'})
        if method != expected_method:
            return reply(HTTPStatus.METHOD_NOT_ALLOWED, {'error': 'Método no permitido.'},
                         [('Allow', expected_method)])
        if path == '/':
            return reply(HTTPStatus.OK, {'corpus': 'NFCorpus', 'endpoints': ['GET /health', 'POST /search'],
                                        'example': {'query': 'plant-based diets', 'model': 'bm25', 'top_k': 10}})
        if path == '/health':
            return reply(HTTPStatus.OK, {'status': 'ok', 'corpus': 'NFCorpus',
                                        'documents': len(indice['documentos']), 'models': list(MODELS)})

        if environ.get('CONTENT_TYPE', '').split(';', 1)[0].strip().lower() != 'application/json':
            return reply(HTTPStatus.UNSUPPORTED_MEDIA_TYPE, {'error': 'Envía application/json.'})
        try:
            body_size = int(environ.get('CONTENT_LENGTH') or '0')
            if not 0 < body_size <= 65536:
                return reply(HTTPStatus.BAD_REQUEST, {'error': 'El cuerpo debe ocupar entre 1 y 65536 bytes.'})
            payload = json.loads(environ['wsgi.input'].read(body_size))
        except (ValueError, RecursionError):
            return reply(HTTPStatus.BAD_REQUEST, {'error': 'JSON inválido.'})
        if not isinstance(payload, dict):
            return reply(HTTPStatus.BAD_REQUEST, {'error': 'Se espera un objeto JSON.'})
        query = payload.get('query')
        model = payload.get('model')
        top_k = payload.get('top_k', 10)
        if not isinstance(query, str) or not query.strip():
            return reply(HTTPStatus.BAD_REQUEST, {'error': 'query debe ser texto no vacío.'})
        if model not in MODELS:
            return reply(HTTPStatus.BAD_REQUEST, {'error': 'model debe ser bm25 o tfidf.'})
        if type(top_k) is not int or top_k < 1:
            return reply(HTTPStatus.BAD_REQUEST, {'error': 'top_k debe ser un entero positivo.'})
        return reply(HTTPStatus.OK, formatear_resultados(query, model, indice, min(top_k, len(indice['documentos']))))

    return application

In [13]:
from io import BytesIO
from wsgiref.util import setup_testing_defaults
from wsgiref.validate import validator


application = validator(create_app(indice))


def request_to_app(method, path, payload=None):
    # Prueba de la aplicación WSGI en memoria. Postman comprueba el servicio mediante HTTP.
    environ = {'REQUEST_METHOD': method, 'PATH_INFO': path, 'SCRIPT_NAME': '', 'QUERY_STRING': ''}
    setup_testing_defaults(environ)
    data = b'' if payload is None else json.dumps(payload).encode('utf-8')
    environ.update(CONTENT_TYPE='application/json', CONTENT_LENGTH=str(len(data)))
    environ['wsgi.input'] = BytesIO(data)
    statuses = []

    def start_response(status, headers, exc_info=None):
        statuses.append(status)

    response = application(environ, start_response)
    try:
        body = json.loads(b''.join(response))
    finally:
        response.close()
    return int(statuses[0].split()[0]), body


status, health = request_to_app('GET', '/health')
assert status == 200 and health['documents'] == len(documentos)
print('GET /health', health)

query = 'effects of green tea on blood pressure'
for model in MODELS:
    status, data = request_to_app('POST', '/search', dict(query=query, model=model, top_k=3))
    rows = data['results']
    assert status == 200 and data['model'] == model and len(rows) == 3
    assert [row['rank'] for row in rows] == [1, 2, 3]
    ids = {row['document_id'] for row in rows}
    assert len(ids) == 3 and ids <= documentos.keys()
    assert rows == sorted(rows, key=lambda row: (-row['score'], row['document_id']))
    print(json.dumps(data, ensure_ascii=False, indent=2))

status, error = request_to_app('POST', '/search', dict(query=query, model='otro', top_k=3))
assert status == 400
print('Modelo desconocido:', status, error)

GET /health {'status': 'ok', 'corpus': 'NFCorpus', 'documents': 3633, 'models': ['tfidf', 'bm25']}
{
  "model": "tfidf",
  "results": [
    {
      "rank": 1,
      "document_id": "MED-5047",
      "score": 0.5871739915538847,
      "title": "The Association between Concentrations of Green Tea and Blood Glucose Levels",
      "snippet": "Our objective was to examine whether habitual green tea consumption is associated with blood glucose levels and other biomarkers of glucose metabolism. We conducted a cross-sectional study of 35 male volunteers, 23–63 years old and…"
    },
    {
      "rank": 2,
      "document_id": "MED-4330",
      "score": 0.5170772118906772,
      "title": "Green and black tea in relation to gynecologic cancers",
      "snippet": "Scope Observational studies have evaluated the relationship between green tea intake and cancers of the ovary and endometrium, but we are not aware of the published studies on green tea intake and risk of human papillomavirus (HPV)-relat

### Evaluar BM25 y TF-IDF con los qrels

Primero comprobamos las fórmulas con un ranking de ejemplo que se puede calcular a mano. Después, la evaluación usa solo las consultas de prueba y los qrels oficiales de NFCorpus.

In [14]:
METRICS = ('Precision@10', 'Recall@10', 'NDCG@10')


def cargar_consultas_y_qrels(documentos):
    """Carga las consultas y los juicios de relevancia oficiales de test."""
    from datasets import load_dataset

    queries = load_dataset('BeIR/nfcorpus', 'queries', split='queries')
    queries_by_id = {row['_id']: row['text'] for row in queries}
    judgments = load_dataset('BeIR/nfcorpus-qrels', split='test')
    qrels = defaultdict(dict)
    for row in judgments:
        query_id, doc_id = row['query-id'], row['corpus-id']
        if query_id not in queries_by_id or doc_id not in documentos:
            raise ValueError(f'Qrel con identificador desconocido: {query_id}, {doc_id}')
        qrels[query_id][doc_id] = int(row['score'])
    if len(qrels) != 323:
        raise ValueError('El número de consultas de test no coincide con BEIR.')
    test_queries = {qid: queries_by_id[qid] for qid in sorted(qrels)}
    return test_queries, dict(qrels)


def calcular_metricas(ranking, relevancia):
    """Calcula Precision@10, Recall@10 y NDCG@10 para una consulta."""
    grades = [relevancia.get(doc_id, 0) for doc_id, _ in ranking[:10]]
    relevant = [grade for grade in relevancia.values() if grade > 0]
    hits = sum(grade > 0 for grade in grades)

    def dcg(values):
        return sum(grade / math.log2(rank + 1) for rank, grade in enumerate(values, 1))

    ideal = dcg(sorted(relevant, reverse=True)[:10])
    return dict(zip(METRICS, (hits / 10, hits / len(relevant) if relevant else 0.0,
                              dcg(grades) / ideal if ideal else 0.0)))

In [15]:
metricas_control = calcular_metricas([('D2', 0.9), ('D1', 0.8)], {'D1': 2, 'D2': 1, 'D3': 1})
assert math.isclose(metricas_control['Precision@10'], 2 / 10)
assert math.isclose(metricas_control['Recall@10'], 2 / 3)
dcg_esperado = 1 + 2 / math.log2(3)
dcg_ideal = 2 + 1 / math.log2(3) + 1 / math.log2(4)
assert math.isclose(metricas_control['NDCG@10'], dcg_esperado / dcg_ideal)
print('Control de las fórmulas:', metricas_control)

Control de las fórmulas: {'Precision@10': 0.2, 'Recall@10': 0.6666666666666666, 'NDCG@10': 0.7224242270408039}


`evaluar_modelos()` ejecuta ambos buscadores sobre las mismas consultas de prueba. Devuelve el detalle por consulta y los promedios por modelo. Las celdas siguientes muestran el resumen y permiten revisar el caso con mayor diferencia de NDCG@10.

In [16]:
def evaluar_modelos(consultas, qrels, indice):
    """Evalúa los dos rankings con los qrels y devuelve métricas por consulta y promedios."""
    detalle = {}
    promedios = {}
    for modelo, buscar in BUSCADORES.items():
        detalle[modelo] = {
            consulta_id: calcular_metricas(buscar(consulta, indice, top_k=10), qrels[consulta_id])
            for consulta_id, consulta in consultas.items()
        }
        promedios[modelo] = {
            metrica: mean(valores[metrica] for valores in detalle[modelo].values())
            for metrica in METRICS
        }
    return detalle, promedios

In [ ]:
from html import escape
from IPython.display import HTML, Markdown, display

test_queries, qrels = cargar_consultas_y_qrels(documentos)
per_query, averages = evaluar_modelos(test_queries, qrels, indice)
print(f"Consultas test evaluadas: {len(test_queries)}")
headers = ' | '.join(METRICS)
result_rows = [f'| Modelo | {headers} |', '| --- | ---: | ---: | ---: |']
for model in MODELS:
    values = ' | '.join(f'{averages[model][metric]:.6f}' for metric in METRICS)
    result_rows.append(f'| {model.upper()} | {values} |')
display(Markdown('\n'.join(result_rows)))

columns = ['ID', 'Consulta', 'Docs relevantes'] + [
    f'{model.upper()} {metric}' for model in MODELS for metric in METRICS
]
table = ['<table><thead><tr>', *(f'<th>{title}</th>' for title in columns), '</tr></thead><tbody>']
for qid, query in test_queries.items():
    row = [escape(qid), escape(query), str(sum(grade > 0 for grade in qrels[qid].values()))]
    row += [f'{per_query[model][qid][metric]:.4f}' for model in MODELS for metric in METRICS]
    table += ['<tr>', *(f'<td>{value}</td>' for value in row), '</tr>']
table.append('</tbody></table>')
display(HTML(f'<details><summary>Ver las {len(test_queries)} consultas test y sus métricas</summary>'
             + ''.join(table) + '</details>'))

In [ ]:
descriptions = {
    'Precision@10': 'proporción de documentos relevantes entre los diez primeros puestos',
    'Recall@10': 'parte del total de documentos relevantes que se recuperó en esos diez puestos',
    'NDCG@10': 'posición de los documentos relevantes ponderada por su grado',
}
comments = []
for metric in METRICS:
    difference = averages['bm25'][metric] - averages['tfidf'][metric]
    if math.isclose(difference, 0, abs_tol=1e-9):
        comments.append(f'- **{metric}.** Los dos modelos obtuvieron el mismo promedio.')
        continue
    winner = 'BM25' if difference > 0 else 'TF-IDF'
    comments.append(f'- **{metric}.** {winner} superó al otro modelo en '
                    f'{abs(difference):.4f} puntos de la escala de 0 a 1. '
                    f'Esta métrica mide la {descriptions[metric]}.')
display(Markdown('\n'.join(comments)))

example_id = max(test_queries, key=lambda qid: abs(
    per_query['bm25'][qid]['NDCG@10'] - per_query['tfidf'][qid]['NDCG@10']
))
example_query = test_queries[example_id]
print(f'Consulta con mayor diferencia absoluta de NDCG@10: {example_id} | {example_query}')
for model in MODELS:
    ranking = BUSCADORES[model](example_query, indice, top_k=10)
    grade_pairs = [(doc_id, qrels[example_id].get(doc_id, 0)) for doc_id, _ in ranking]
    print(f'{model.upper()} · NDCG@10 = {per_query[model][example_id]["NDCG@10"]:.4f}')
    print('  Resultados en orden (ID, grado):', grade_pairs)
display(Markdown('Un grado alto tiene mayor efecto en NDCG cuando aparece en los primeros puestos. '
                 'BM25 satura la frecuencia y ajusta por longitud. TF-IDF pondera la frecuencia '
                 'linealmente y usa la norma del vector completo. Estas diferencias pueden alterar '
                 'el orden incluso cuando los modelos recuperan varios documentos en común.'))

### Publicación y prueba en Postman

1. Extraer `NFCorpus_servicio.zip`. El archivo `README.md` describe la publicación como Space Docker público o en otro hospedaje compatible con Docker. El servicio carga el corpus de Hugging Face al iniciar.
2. Importar `NFCorpus.postman_collection.json` en Postman y configurar `base_url` con la URL pública. Para una prueba local, usar `http://127.0.0.1:7860`.
3. Enviar `GET /health` y las dos solicitudes `POST /search` de la colección. Se puede cambiar el texto de `query` libremente en **Body → raw → JSON**.

**Estado de entrega.** Las celdas de evaluación requieren acceso a Hugging Face para guardar las métricas. El paquete incluye la configuración de despliegue para obtener una URL pública.

### Cómo hacer la petición POST en Postman

La API del examen está publicada en Render. Primero comprueba [GET /health](https://nfcorpus-api.onrender.com/health). La respuesta esperada indica `status: ok`, `corpus: NFCorpus` y `documents: 3633`.

1. En Postman, abre **New → HTTP Request** y selecciona el método **POST**.
2. Introduce la URL `https://nfcorpus-api.onrender.com/search`.
3. En **Body**, selecciona **raw** y luego **JSON**. En **Headers** debe figurar `Content-Type: application/json`.
4. Pega este cuerpo y pulsa **Send**.

```json
{
  "query": "effects of green tea on blood pressure",
  "model": "bm25",
  "top_k": 3
}
```

5. Comprueba que Postman muestre **200 OK**. La respuesta incluye `model` y una lista `results` ordenada por puntaje. Cada resultado incluye `rank`, `document_id`, `score`, `title` y `snippet`.
6. Cambia `"bm25"` por `"tfidf"` y vuelve a pulsar **Send**. Mantén el mismo `query` y `top_k` para comparar las posiciones. En las pruebas contra Render, las dos solicitudes devolvieron **200 OK** y tres resultados. Si la instancia gratuita está inactiva, la primera solicitud puede tardar mientras se reactiva.

La misma solicitud se puede ejecutar importando `NFCorpus.postman_collection.json` y asignando `https://nfcorpus-api.onrender.com` a la variable `base_url`.